## Loading Data

In [8]:
import pandas as pd

data = pd.read_csv("/Users/tanapondtriyanond/Desktop/CS74-Final-Project-MoralBERT/all_data.csv")

print(data.shape)
print(data.columns)
data.head()

(1999516, 46)
Index(['id', 'comment_text', 'split', 'created_date', 'publication_id',
       'parent_id', 'article_id', 'rating', 'funny', 'wow', 'sad', 'likes',
       'disagree', 'toxicity', 'severe_toxicity', 'obscene', 'sexual_explicit',
       'identity_attack', 'insult', 'threat', 'male', 'female', 'transgender',
       'other_gender', 'heterosexual', 'homosexual_gay_or_lesbian', 'bisexual',
       'other_sexual_orientation', 'christian', 'jewish', 'muslim', 'hindu',
       'buddhist', 'atheist', 'other_religion', 'black', 'white', 'asian',
       'latino', 'other_race_or_ethnicity', 'physical_disability',
       'intellectual_or_learning_disability', 'psychiatric_or_mental_illness',
       'other_disability', 'identity_annotator_count',
       'toxicity_annotator_count'],
      dtype='str')


,id,comment_text,split,created_date,publication_id,parent_id,article_id,rating,funny,wow,...,white,asian,latino,other_race_or_ethnicity,physical_disability,intellectual_or_learning_disability,psychiatric_or_mental_illness,other_disability,identity_annotator_count,toxicity_annotator_count
0,1083994,He got his money... now he lies in wait till a...,train,2017-03-06 15:21:53.675241+00,21,NaN,317120,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,67
1,650904,Mad dog will surely put the liberals in mental...,train,2016-12-02 16:44:21.329535+00,21,NaN,154086,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,76
2,5902188,And Trump continues his lifelong cowardice by ...,train,2017-09-05 19:05:32.341360+00,55,NaN,374342,approved,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,63
3,7084460,"""while arresting a man for resisting arrest"".\...",test,2016-11-01 16:53:33.561631+00,13,NaN,149218,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,76
4,5410943,Tucker and Paul are both total bad ass mofo's.,train,2017-06-14 05:08:21.997315+00,21,NaN,344096,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,80


In [9]:
# 2M rows of data
print(len(data))

# Data that has identity annotators and identity tags
labeled = data[data["identity_annotator_count"] > 0]
print(len(labeled))

1999516
448000


## Clean Data

In [10]:
# make copy to clean data
df = data.copy()

# Check missing data in each column
print(data.comment_text.isna().sum())

#remove rows with no comment_text
df = df.dropna(subset=["comment_text"])

#remove rows with duplicate text
print(data.comment_text.duplicated().sum())
df = df.drop_duplicates(subset=["comment_text"])

print(df.comment_text.isna().sum())
print(df.comment_text.duplicated().sum())


4
27601
0
0


## Toxicity Samples

Sorting comments into 4 toxicity levels, then taking 5,000 from each. If we don't have bins then most of the data could be labeled clean comments, our results would be skewed.

In [11]:
# Get a sample from each toxicity level
pieces = []

print(df.shape)

#remove rows with identity annotation
id_cols = ['male', 'female', 'transgender', 'other_gender', 'heterosexual',
           'homosexual_gay_or_lesbian', 'bisexual', 'other_sexual_orientation',
           'christian', 'jewish', 'muslim', 'hindu', 'buddhist', 'atheist',
           'other_religion', 'black', 'white', 'asian', 'latino',
           'other_race_or_ethnicity', 'physical_disability',
           'intellectual_or_learning_disability', 'psychiatric_or_mental_illness',
           'other_disability']

tox_df = df[df[id_cols].isna().all(axis=1) & (df["identity_annotator_count"].fillna(0) == 0)].copy()

print(tox_df.shape)

# Making the toxicity levels
tox_df["tox_bin"] = pd.cut(tox_df["toxicity"], bins=[-0.01, 0, 0.2, 0.5, 1], labels=['clean', 'low', 'mid', 'toxic'])
tox_df.tox_bin.value_counts()
    
for level, group in tox_df.groupby("tox_bin", observed=True): # level = toxicity level, group are the rows in that level
    n = min(len(group), 5000) #take all rows or 5,000
    pieces.append(group.sample(n, random_state=42))

toxicity_samples = pd.concat(pieces)
print(toxicity_samples["tox_bin"].value_counts())

toxicity_samples.to_csv("toxicity_samples.csv", index=False)

(1971914, 46)
(1529361, 46)
tox_bin
clean    5000
low      5000
mid      5000
toxic    5000
Name: count, dtype: int64


## Identity Row Samples

Getting rows that have an identity_annotator_count for experiments.

In [12]:
#keep only rows that were labeled for identity
identity_rows = df[df["identity_annotator_count"] > 0].copy()
print("labeled rows:", len(identity_rows))

#all 24 identity groups
identity_groups = ['male', 'female', 'transgender', 'other_gender',
                   'heterosexual', 'homosexual_gay_or_lesbian', 'bisexual',
                   'other_sexual_orientation', 'christian', 'jewish', 'muslim',
                   'hindu', 'buddhist', 'atheist', 'other_religion', 'black',
                   'white', 'asian', 'latino', 'other_race_or_ethnicity',
                   'physical_disability', 'intellectual_or_learning_disability',
                   'psychiatric_or_mental_illness', 'other_disability']

# mark whether a comment mentions each group
# (threshold >= 0.5: at least half of the raters said the comment mentions that group)
for g in identity_groups:
    identity_rows["has_" + g] = identity_rows[g] >= 0.5

# count how many comments mention each group
has_cols = ["has_" + g for g in identity_groups]
counts = identity_rows[has_cols].sum().sort_values(ascending=False)
print(counts)

# keep groups that are not "other_" and have at least 100 comments for experimenting
keep_groups = []
for g in identity_groups:
    if g.startswith("other_"):
        continue
    if counts["has_" + g] >= 100:
        keep_groups.append(g)

print("groups kept:", keep_groups)

labeled rows: 442553
has_female                                 58267
has_male                                   48611
has_christian                              44485
has_white                                  27254
has_muslim                                 22819
has_black                                  16298
has_homosexual_gay_or_lesbian              11995
has_jewish                                  8416
has_psychiatric_or_mental_illness           5362
has_asian                                   4998
has_transgender                             2729
has_latino                                  2217
has_atheist                                 1686
has_heterosexual                            1428
has_buddhist                                 634
has_hindu                                    630
has_other_race_or_ethnicity                  542
has_other_religion                           351
has_bisexual                                 318
has_intellectual_or_learning_disability      117

In [13]:
identity_rows.head()

,id,comment_text,split,created_date,publication_id,parent_id,article_id,rating,funny,wow,...,has_other_religion,has_black,has_white,has_asian,has_latino,has_other_race_or_ethnicity,has_physical_disability,has_intellectual_or_learning_disability,has_psychiatric_or_mental_illness,has_other_disability
7678,627762,OH yes - Were those evil Christian Missionarie...,train,2016-11-26 15:56:03.862109+00,13,627198.0,152737,approved,0,0,...,False,False,False,False,False,False,False,False,False,False
7679,5892815,Why is this black racist crap still on the G&M...,train,2017-09-03 23:20:08.226613+00,54,NaN,373428,rejected,0,0,...,False,True,True,False,False,False,False,False,False,False
7680,416437,even up here.......BLACKS!,train,2016-08-04 16:48:07.175252+00,21,NaN,143025,rejected,0,0,...,False,True,False,False,False,False,False,False,False,False
7681,5137126,Blame men. There's always an excuse to blame ...,train,2017-04-15 19:00:45.032674+00,54,5136907.0,327125,rejected,0,0,...,False,False,False,False,False,False,False,False,False,False
7682,855753,And the woman exposing herself saying grab thi...,train,2017-01-18 01:50:57.478867+00,13,849081.0,162008,rejected,0,0,...,False,False,False,False,False,False,False,False,False,False


In [14]:
# Getting samples for identity experiment

# Count how many groups each comment mentions (a single comment can mention many identity groups)
identity_rows["n_groups"] = identity_rows[has_cols].sum(axis=1)

# We only want comments that mention exactly one group, and comments that mention none (we drop comments that have more than 1 group because it would be hard to experiment)
single = identity_rows[identity_rows["n_groups"] == 1].copy()
control = identity_rows[identity_rows["n_groups"] == 0].copy()

# name the group
single["group"] = single[has_cols].idxmax(axis=1).str.replace("has_", "")
control["group"] = "none"

# keep only the groups we chose in Step 4
single = single[single["group"].isin(keep_groups)]

print(single["group"].value_counts())   # check sizes after "exactly one"

# sample up to 500 per group
pieces = []
for name, group in pd.concat([single, control]).groupby("group"):
    n = min(len(group), 3000)
    pieces.append(group.sample(n, random_state=42))

identity_samples = pd.concat(pieces)
print(identity_samples["group"].value_counts())

# save data to file
identity_samples.to_csv("identity_samples.csv", index=False)

group
female                                 30212
christian                              29292
male                                   18498
muslim                                 13710
white                                  12594
black                                   6400
homosexual_gay_or_lesbian               5292
psychiatric_or_mental_illness           4297
jewish                                  3631
asian                                   3400
transgender                             1271
atheist                                  855
latino                                   803
heterosexual                             172
hindu                                    120
buddhist                                 118
intellectual_or_learning_disability       93
bisexual                                  30
Name: count, dtype: int64
group
asian                                  3000
black                                  3000
christian                              3000
female              